In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Cargamos los datos ORIGINALES (crudos) para probar el pipeline desde cero
df = pd.read_csv('./data/titanic.csv')
X = df.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# Transformaciones para números (Edad y Tarifa)
numeric_features = ['Age', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Llena nulos
    ('scaler', StandardScaler())                  # Escala datos
])

# Transformaciones para texto (Sexo y Puerto)
categorical_features = ['Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Llena nulos
    ('encoder', OneHotEncoder(handle_unknown='ignore'))   # Convierte a números
])

# Combinamos ambos en un solo "Preprocesador"
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [3]:
# El Pipeline une el preprocesador con el modelo ganador (Random Forest)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

# ¡Entrenamos todo el sistema de un solo golpe!
full_pipeline.fit(X_train, y_train)

# Evaluamos
score = full_pipeline.score(X_test, y_test)
print(f"Precisión del Pipeline completo: {score:.4f}")

Precisión del Pipeline completo: 0.7654


In [4]:
# Guardamos el pipeline completo. 
# Ahora el chatbot solo necesita cargar este archivo para limpiar y predecir.
with open('pipeline_titanic.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

print("¡Arquitectura completada! El archivo pipeline_titanic.pkl está listo para producción.")

¡Arquitectura completada! El archivo pipeline_titanic.pkl está listo para producción.
